In [1]:
import polars as pl 
import polars_ds as pds 
import requests 
import json 
import duckdb
from pathlib import Path
from src.utils import scrape_ticket_sections, get_available_events, scrape_match_results, scrape_eliteserien_results,create_table_after_round

In [2]:
# List all table names
db_path = Path("data/brann.duckdb")
con = duckdb.connect(str(db_path))
table_names = con.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'main'").fetchall()
print("Available tables:")
for table in table_names:
    print(f"  - {table[0]}")
con.close()

Available tables:
  - dim_teams
  - fct_goal_scorers
  - fct_league_standings
  - fct_matches
  - raw_eliteserien_goal_scorers
  - raw_eliteserien_results


In [25]:
con = duckdb.connect('data/brann.duckdb')
goal_scorers = pl.from_arrow(con.execute(
    """
    SELECT *
FROM fct_goal_scorers
""").arrow())
con.close()
goal_scorers


season,date,matchday,home_team,away_team,result,scorer_team,scorer_name,ingested_at
i64,date,i64,str,str,str,str,str,datetime[μs]
2015,2015-04-06,1,"""Sandefjord""","""Bodø/Glimt""","""3:1""","""FK Bodø/Glimt""","""H. Furebotn""",2026-09-04 14:29:08.523745
2015,2015-04-06,1,"""Mjøndalen""","""Viking FK""","""1:0""","""Mjøndalen IF""","""S. Kapidzic""",2026-09-04 14:29:08.523745
2015,2015-04-06,1,"""Sandefjord""","""Bodø/Glimt""","""3:1""","""Sandefjord Fotball""","""A. Gabrielsen""",2026-09-04 14:29:08.523745
2015,2015-04-06,1,"""Sandefjord""","""Bodø/Glimt""","""3:1""","""Sandefjord Fotball""","""J. Mendy""",2026-09-04 14:29:08.523745
2015,2015-04-06,1,"""Sandefjord""","""Bodø/Glimt""","""3:1""","""Sandefjord Fotball""","""J. Mendy""",2026-09-04 14:29:08.523745
…,…,…,…,…,…,…,…,…
2026,2026-09-06,20,"""Molde FK""","""KFUM Oslo""","""2:0""","""Molde FK""","""T. Koné-Doherty""",2026-09-08 09:13:42.536941
2026,2026-09-06,20,"""Sarpsborg 08""","""Vålerenga""","""2:2""","""Sarpsborg 08 FF""","""C. Niyukuri""",2026-09-08 09:13:42.536941
2026,2026-09-06,20,"""Sarpsborg 08""","""Vålerenga""","""2:2""","""Sarpsborg 08 FF""","""S. Sørli""",2026-09-08 09:13:42.536941


In [2]:
con = duckdb.connect('data/brann.duckdb')
latest = pl.from_arrow(con.execute("""
    SELECT * FROM fct_league_standings 
""").arrow())
con.close()

latest

season,matchday,team,total_points,total_goals_for,total_goals_against,goal_difference,position
i64,i64,str,"decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]",i64
2026,19,"""Bodø/Glimt""",44,47,14,33,1
2026,19,"""Viking FK""",43,42,18,24,2
2026,19,"""Tromsø IL""",35,34,20,14,3
2026,19,"""Molde FK""",30,36,29,7,4
2026,19,"""SK Brann""",26,36,27,9,5
…,…,…,…,…,…,…,…
2015,1,"""Viking FK""",0,0,1,-1,12
2015,1,"""Tromsø IL""",0,0,1,-1,13
2015,1,"""Bodø/Glimt""",0,1,3,-2,14


In [3]:


# Connect to DuckDB and read raw_eliteserien_results
db_path = Path("data/brann.duckdb")
con = duckdb.connect(str(db_path))
eliteserien_db = pl.from_arrow(con.execute("SELECT * FROM dim_teams").arrow())
con.close()

print(f"Loaded {len(eliteserien_db)} records from DuckDB")
eliteserien_db

Loaded 192 records from DuckDB


season,team_name
i64,str
2015,"""Aalesunds FK"""
2015,"""Bodø/Glimt"""
2015,"""Haugesund"""
2015,"""IK Start"""
2015,"""Lillestrøm SK"""
…,…
2026,"""Sandefjord"""
2026,"""Sarpsborg 08"""
2026,"""Tromsø IL"""


In [ ]:
from src.config import ELITESERIEN_SEASONS, SCRAPE_DELAY_SECONDS
from src.utils import scrape_eliteserien_goal_scorers_for_seasons

goal_scorers = scrape_eliteserien_goal_scorers_for_seasons(
    seasons=[(2014,2015)],
    delay_seconds=1,
)

processing match 1 of 240
processing match 2 of 240
processing match 3 of 240
processing match 4 of 240
processing match 5 of 240
processing match 6 of 240
processing match 7 of 240
processing match 8 of 240
processing match 9 of 240


In [3]:
import polars as pl
pl.DataFrame(goal_scorers)

season,date,matchday,home_team,away_team,result,scorer_team,scorer_name
i64,date,i64,str,str,str,str,str
2015,2015-04-06,1,"""Mjøndalen""","""Viking FK""","""1:0""","""Mjøndalen IF""","""S. Kapidzic"""
2015,2015-04-06,1,"""Rosenborg BK""","""Aalesunds FK""","""5:0""",""" ""","""P. Helland"""
2015,2015-04-06,1,"""Rosenborg BK""","""Aalesunds FK""","""5:0""",""" ""","""P. Helland"""
2015,2015-04-06,1,"""Rosenborg BK""","""Aalesunds FK""","""5:0""",""" ""","""A. Søderlund"""
2015,2015-04-06,1,"""Rosenborg BK""","""Aalesunds FK""","""5:0""",""" ""","""A. Søderlund"""
…,…,…,…,…,…,…,…
2026,2026-08-30,19,"""Lillestrøm SK""","""Fredrikstad FK""","""1:4""","""Lillestrøm SK""","""F. Gulbrandsen"""
2026,2026-08-30,19,"""Lillestrøm SK""","""Fredrikstad FK""","""1:4""","""Fredrikstad FK""","""S. Owusu"""
2026,2026-08-30,19,"""Lillestrøm SK""","""Fredrikstad FK""","""1:4""","""Fredrikstad FK""","""M. Nilsson"""


In [2]:
available_events = get_available_events()

In [3]:
for event in available_events:
    print(event['event_id'])


1085523
1187151
1188514


In [18]:
eliteserien_results = scrape_eliteserien_results(season_id = 2025,year = 2026)

In [19]:
pl.DataFrame(eliteserien_results)

date,matchday,home_team,away_team,result,snapshot_at
date,i64,str,str,str,"datetime[μs, UTC]"
2026-03-14,1,"""HamKam""","""Viking FK""","""2:1""",2026-09-01 09:53:45.721672 UTC
2026-03-14,1,"""Molde FK""","""Rosenborg BK""","""2:0""",2026-09-01 09:53:45.721672 UTC
2026-03-15,1,"""Kristiansund BK""","""SK Brann""","""3:2""",2026-09-01 09:53:45.721672 UTC
2026-03-15,1,"""KFUM Oslo""","""IK Start""","""2:0""",2026-09-01 09:53:45.721672 UTC
2026-03-15,1,"""Vålerenga""","""Sandefjord""","""1:0""",2026-09-01 09:53:45.721672 UTC
…,…,…,…,…,…
2026-08-30,19,"""IK Start""","""KFUM Oslo""","""4:1""",2026-09-01 09:53:45.721672 UTC
2026-08-30,19,"""Tromsø IL""","""Sarpsborg 08""","""0:0""",2026-09-01 09:53:45.721672 UTC
2026-08-30,19,"""Viking FK""","""Aalesunds FK""","""2:1""",2026-09-01 09:53:45.721672 UTC


In [3]:
create_table_after_round(eliteserien_results)

matchday,team,points,goals_for,goals_against,goal_difference,total_points,total_goals_for,total_goals_against,total_goal_difference,table_position
i64,str,i32,i64,i64,i64,i32,i64,i64,i64,u32
1,"""HamKam""",3,2,1,1,3,2,1,1,1
1,"""KFUM Oslo""",3,2,0,2,3,2,0,2,1
1,"""Kristiansund BK""",3,3,2,1,3,3,2,1,1
1,"""Lillestrøm SK""",3,3,1,2,3,3,1,2,1
1,"""Molde FK""",3,2,0,2,3,2,0,2,1
…,…,…,…,…,…,…,…,…,…,…
18,"""KFUM Oslo""",1,1,1,0,19,19,27,-8,12
18,"""Sandefjord""",3,2,1,1,18,15,23,-8,13
18,"""Aalesunds FK""",1,5,5,0,15,27,41,-14,14


In [4]:
paok = scrape_ticket_sections(1187151)

In [8]:
from src.utils import scrape_eliteserien_results_for_seasons

results = scrape_eliteserien_results_for_seasons(
    seasons=[
        (2025, 2026),
        (2024, 2025),
        (2023, 2024),
    ],
    delay_seconds=5.0,
)

In [10]:
pl.DataFrame(results).filter(pl.col('date').dt.year()==2025).filter(pl.col('home_team').str.contains('Brann'))

date,matchday,home_team,away_team,result,snapshot_at
date,i64,str,str,str,"datetime[μs, UTC]"
2025-04-06,2,"""SK Brann""","""Tromsø IL""","""3:1""",2026-09-02 11:00:27.335750 UTC
2025-04-10,17,"""SK Brann""","""Strømsgodset""","""2:1""",2026-09-02 11:00:27.335750 UTC
2025-04-27,4,"""SK Brann""","""Bryne""","""3:2""",2026-09-02 11:00:27.335750 UTC
2025-05-11,6,"""SK Brann""","""Rosenborg BK""","""0:0""",2026-09-02 11:00:27.335750 UTC
2025-05-16,7,"""SK Brann""","""Sarpsborg 08""","""2:2""",2026-09-02 11:00:27.335750 UTC
…,…,…,…,…,…
2025-09-28,19,"""SK Brann""","""Fredrikstad FK""","""1:0""",2026-09-02 11:00:27.335750 UTC
2025-10-18,25,"""SK Brann""","""Haugesund""","""4:1""",2026-09-02 11:00:27.335750 UTC
2025-10-29,23,"""SK Brann""","""Bodø/Glimt""","""1:2""",2026-09-02 11:00:27.335750 UTC


In [2]:
from src.agent import run_question

run_question("Hva var Brann sin forrige bortekamp?")

BinderException: Binder Error: Ambiguous reference to column name "season" (use: "fct_matches.season" or "last_season.season")

LINE 6:         season,
                ^

In [3]:

import src.agent
for event in src.agent.agent.stream(
    {"question": "Hva var Brann sin forrige bortekamp?"},
    stream_mode="updates",
):
    print(event)

{'generate_sql': {'sql': "WITH last_season AS (\n    SELECT MAX(season) AS season FROM fct_matches\n),\nlast_away_match AS (\n    SELECT \n        season,\n        date,\n        matchday,\n        home_team,\n        away_team,\n        result,\n        winner\n    FROM fct_matches, last_season\n    WHERE season = last_season.season\n      AND away_team = 'SK Brann'\n    ORDER BY date DESC\n    LIMIT 1\n)\nSELECT \n    season,\n    date,\n    matchday,\n    home_team,\n    away_team,\n    result,\n    winner\nFROM last_away_match;"}}


BinderException: Binder Error: Ambiguous reference to column name "season" (use: "fct_matches.season" or "last_season.season")

LINE 6:         season,
                ^

In [35]:
query = """
WITH last_season AS (
    SELECT MAX(season) AS season
    FROM fct_matches
),
last_away_match AS (
    SELECT
        fct_matches.season,
        date,
        matchday,
        home_team,
        away_team,
        result,
        winner
    FROM fct_matches
    CROSS JOIN last_season
    WHERE fct_matches.season = last_season.season
      AND away_team = 'SK Brann'
    ORDER BY date DESC
    LIMIT 1
)
SELECT *
FROM last_away_match;
"""

con = duckdb.connect('data/brann.duckdb')
goal_scorers = pl.from_arrow(con.execute(
   query).arrow())
con.close()
goal_scorers

season,date,matchday,home_team,away_team,result,winner
i64,date,i64,str,str,str,str
2026,2026-08-30,19,"""Sandefjord""","""SK Brann""","""0:0""","""draw"""
